# Reach, the logit and selection

tri answers a build in one pass. Hard constraints (requirements, prerequisites, exclusions, the talent cap $76-2m$ and the mantra slots) and authored rules filter or force items. Every other item $t$ gets one logit

$$z_t = \alpha_0 + \alpha_L L_t + \alpha_q \operatorname{logit} q_t + \alpha_D D_t + \alpha_P P_t,$$

and slots are filled by $z$. $L_t$ is the field logit at the build's point, $q_t$ the pick rate among its nearest builds (reviewed builds included), $D_t$ the direct review evidence and $P_t$ the propagated evidence (off by default).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
from tri.api import TriConfig

cfg = TriConfig()
L, S = cfg.ledger, cfg.infer               # similarity levels and scorer constants of this version
h = L.reach / np.sqrt(-2 * np.log(L.sim_min))
print(f"reach {L.reach}, sim_min {L.sim_min}, sim_full {L.sim_full}, sim_same {L.sim_same}, h = {h:.3f}")
print({k: v for k, v in vars(S).items() if k.startswith("sc_")})

## Directed reach

For an add or remove whose reason is about the item itself (necessary for archetype, generally good, no synergy, scenario mismatch, broken), reach is directed. With $d = u_b - u_r$ and $g_t$ the unit gradient of $L_t$,

$$\pi = d\cdot g_t, \qquad K_c(b) = \chi\,\exp\!\left(-\frac{\pi^2}{2h_\parallel^2}\right)\exp\!\left(-\frac{\lVert d\rVert^2-\pi^2}{2h_\perp^2}\right).$$

The kernel is narrow along the direction in which the item's value changes and wide across it: moving along $g_t$ changes whether the item is worth it, moving across it does not. Every other claim uses $K_c = \sigma$. In both cases $K \ge \sigma_{\text{full}}$ counts as 1 and $K < \sigma_{\min}$ as 0.

In [ ]:
def clip_k(k):
    return np.where(k >= L.sim_full, 1.0, np.where(k < L.sim_min, 0.0, k))

def directed(angle=30, h_par=float(S.sc_h_par), h_perp=float(S.sc_h_perp), directed=True):
    g = np.linspace(-6, 6, 241)
    X, Y = np.meshgrid(g, g)
    gt = np.array([np.cos(np.radians(angle)), np.sin(np.radians(angle))])
    pi = X * gt[0] + Y * gt[1]
    d2 = X ** 2 + Y ** 2
    k = np.exp(-pi ** 2 / (2 * h_par ** 2) - (d2 - pi ** 2) / (2 * h_perp ** 2)) if directed else np.exp(-d2 / (2 * h ** 2))
    kk = clip_k(k)
    fig, ax = plt.subplots(figsize=(5.4, 5))
    ax.imshow(kk, extent=(-6, 6, -6, 6), origin="lower", vmin=0, vmax=1, cmap="viridis")
    ax.arrow(0, 0, 2 * gt[0], 2 * gt[1], width=0.06, color="w")
    ax.text(2.2 * gt[0], 2.2 * gt[1], r"$g_t$", color="w")
    ax.set_title(f"reach area {kk.sum() * (g[1] - g[0]) ** 2:.1f} (after the 0/1 clipping)")
    plt.show()

interact(directed, angle=IntSlider(30, 0, 180), h_par=FloatSlider(value=float(S.sc_h_par), min=0.5, max=5, step=0.05),
         h_perp=FloatSlider(value=float(S.sc_h_perp), min=0.5, max=8, step=0.05), directed=Checkbox(True));

## Direct evidence and the logit

$$D_t = \sum_{c \text{ on } t} \operatorname{sign}(c)\, w_c\, K_c(b)\, \kappa_c,$$

with sign $+1$ for add and keep, $-1$ for remove, and $\kappa_c = \alpha_K$ for keep claims, 1 otherwise. Claims combine by weight times reach.

Selection then reads $z_t$ against fixed thresholds: an unprotected pick with $z_t < \theta_-$ is removed; a review fills only with items scoring at least $\theta_+$ (and keeps the input's counts), a completion fills every slot; an outside item replaces the weakest removable pick when it scores more than $\delta$ higher (at most 3 swaps).

The toy below is one item with a neighbour rate $q$, an older add claim from a reviewed build at similarity $\sigma_1$ and a newer remove at $\sigma_2$ whose two reviewed builds are $\sigma_{12}$ apart.

In [ ]:
def logit(p):
    p = min(max(p, 1e-4), 1 - 1e-4)
    return np.log(p / (1 - p))

def f(s):
    return 0.0 if s >= L.sim_full else ((L.sim_full - s) / (L.sim_full - L.sim_min) if s >= L.sim_min else 1.0)

def item(q=0.3, s_add=0.97, s_rem=0.85, s_between=0.9, keep_too=False, in_build=True):
    k = lambda s: 1.0 if s >= L.sim_full else (0.0 if s < L.sim_min else s)
    w_add = f(s_between)                                   # the newer remove supersedes the older add
    D = w_add * k(s_add) - 1.0 * k(s_rem) + (S.sc_keep * 1.0 if keep_too else 0.0)
    z = S.sc_bias + S.sc_nbr * logit(q) + S.sc_direct * D
    if in_build:
        verdict = "removed" if z < S.sc_remove else "kept"
    else:
        verdict = "added by a review" if z >= S.sc_fill else "not added by a review (a completion may still fill it)"
    print(f"older add weight w = {w_add:.2f}, D = {D:+.2f}, z = {z:+.2f}  ->  {verdict}")
    print(f"thresholds: remove below {S.sc_remove}, review fill from {S.sc_fill}, swap margin {S.sc_swap}")

interact(item, q=FloatSlider(value=0.3, min=0.01, max=0.99, step=0.01),
         s_add=FloatSlider(value=0.97, min=0.5, max=1, step=0.005), s_rem=FloatSlider(value=0.85, min=0.5, max=1, step=0.005),
         s_between=FloatSlider(value=0.9, min=0.5, max=1, step=0.005), keep_too=Checkbox(False), in_build=Checkbox(True));

## Binding claims

A claim with $w_c \ge 0.5$ overrides the scorer when it is on the build itself ($\sigma \ge \sigma_{\text{same}}$), is a broken removal, is a strong generally-good add, is a strong add or remove with a non-relative reason from a build at $\sigma \ge 0.95$, or is a sidegrade mark. Because the build itself always binds, every review reproduces on its own build at once.

## Selection order

1. Game constraints: Vow of Iron, weapon rules, never rules and mutual exclusions (of two exclusive picks the lower $z$ goes).
2. Binding removals; a "needs" removal adds the needed picks if there is room and removes otherwise.
3. Game limits, then forced picks: an input over the talent cap $76-2m$ or its mantra slots first loses its lowest-$z$ unprotected picks; prerequisites in use are never removed. Forced items go in with their prerequisite chains first.
4. Removals below $\theta_-$, except prerequisites, slot grants in use and the only dependant of a picked prerequisite.
5. Fill by $z$: mantras first (talents first on Silentheart), then forced talents, then talents. Wildcard slots take any mantra type.
6. Up to 3 swaps with margin $\delta$.

A sidegrade side is committed as soon as one of its items is picked, so the two sides are never blended, and removed items are never re-added.

## Propagation and categories

Propagated evidence nudges unclaimed similar items, $P_s = \sum_c \operatorname{sign}(c)\, w_c K_c \max(0, \cos(s,t)-\tau)/(1-\tau)$ over archetype-fit claims, with $\cos$ between mean-centred worth vectors. It gained nothing measurable, so `sc_propagate` is off.

`Tri.categories()` reports foundational, central or luxury: the admin category first, then the pass of an authored take rule, then foundational for a talent taken by at least 90% of the builds that can take it, otherwise luxury. Only admin categories and authored rules force picks.